#### Skip Connetion and Residual Block

1. The Skip Connection (The Shortcut)
    - Imagine a traditional neural network as a strict assembly line. A piece of data (the input) enters Station 1, gets modified, moves to Station 2, gets modified again, and so on. Every single piece of data must go through every single station.

    - A skip connection builds an alternative conveyor belt that bypasses a few stations.

    - Instead of forcing the data to only travel through the heavy processing layers, a skip connection takes a copy of the original input data, leaps over a couple of layers, and adds it directly to the output of those layers further down the line.

2. The Residual Block
    - A residual block is simply a small chunk of a neural network (usually two or three convolutional layers) that has a skip connection wrapped around it. 

3. ResNets are built by stacking dozens or hundreds of these blocks on top of each other.

#### ResNets

- ResNets are CNNs. "ResNet" is just the specific name for a Convolutional Neural Network that uses skip connections. If you strip away the + x skip connections from the code we wrote, you are left with a perfectly normal, standard CNN (often called a "Plain CNN" in research papers).

1. Phase 1: The "Stem"
    - Before the data even sees a residual block, it passes through the stem. The goal here is rapid downsampling. High-resolution images (like $224 \times 224$ pixels) take up a massive amount of memory.
    
    - The stem uses a heavy traditional convolution (usually a $7 \times 7$ kernel with a stride of 2) followed by a Max Pooling layer. This rapidly shrinks the image down to a quarter of its original size while extracting the most basic, low-level features like edges and colors.
    
2. Phase 2: The "Stages" (The Body of the Network)
    - This is where the residual blocks live. The body is typically divided into four stages (often coded as layer1, layer2, layer3, and layer4 in PyTorch).
    
        - Stage 1: Receives the output from the stem. It stacks a few residual blocks together. The spatial dimensions and the number of channels stay exactly the same throughout this entire stage.Stage 2: This is where the shift happens. The very first residual block in 
        
        - Stage 2 does two things simultaneously:
            - It uses a stride of 2 in its first convolution, which cuts the image's height and width in half (e.g., from $56 \times 56$ to $28 \times 28$).
            
            - It doubles the number of filters (channels) from 64 to 128.Because the input and output shapes no longer match, this is exactly where that $1 \times 1$ skip connection we discussed earlier kicks in to project the input to the new, deeper shape. After this transition block, the remaining blocks in Stage 2 keep the dimensions stable.
        
        - Stages 3 & 4: These stages repeat the exact same pattern as Stage 2. The first block halves the resolution and doubles the channels, and the subsequent blocks process the data at that new shape.
    
    - By the end of Stage 4, your original $224 \times 224 \times 3$ image has been transformed into a dense $7 \times 7$ grid containing 512 distinct feature channels.

3. Phase 3: The "Head" (Classification)

    - At this point, you have a $7 \times 7$ grid of features, but you need a single prediction (like "is this a dog?").
    
    - Instead of flattening the $7 \times 7$ grid into a massive, memory-heavy array (which is what older networks like VGG did), ResNets use Global Average Pooling. This takes the average value of each $7 \times 7$ feature map, squashing the tensor down to a $1 \times 1 \times 512$ vector.
    
    - Finally, that vector is passed through a single fully connected (Linear) layer to output the final class probabilities.

In [12]:
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torchvision import datasets, transforms
# from torch.utils.data import random_split, DataLoader

# # 1. Prepare the data
# transform = transforms.Compose([
#     transforms.ToTensor(),
#     transforms.Normalize((0.1307,), (0.3081,))
# ])

# full_train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
# test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# # 2. Split into Train and Validation
# train_dataset, val_dataset = random_split(full_train_dataset, [0.8, 0.2])

# # 3. Create DataLoaders
# batch_size = 64
# trainloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# valloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
# testloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import random_split, DataLoader # <--- Don't forget to import these!

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),  # per-channel mean
                         (0.5, 0.5, 0.5))  # per-channel std
])

# 1. Load the FULL training dataset
full_trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)

# Extract the classes from the FULL dataset before splitting it
classes = full_trainset.classes  # ['airplane', 'automobile', 'bird', ...]

# 2. Split into Train and Validation using percentages (80% Train, 20% Val)
trainset, valset = random_split(full_trainset, [0.8, 0.2])

# 3. Create DataLoaders for Train and Validation
trainloader = DataLoader(
    trainset, batch_size=64, shuffle=True, num_workers=2
)
valloader = DataLoader(
    valset, batch_size=64, shuffle=False, num_workers=2
)

# 4. Load the Test dataset and create its DataLoader
testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform
)
testloader = DataLoader(
    testset, batch_size=64, shuffle=False, num_workers=2
)

print(f"Total training images: {len(full_trainset)}")
print(f"--> Sliced into: {len(trainset)} Train | {len(valset)} Validation")
print(f"Total testing images: {len(testset)}")

Using device: cuda
Total training images: 50000
--> Sliced into: 40000 Train | 10000 Validation
Total testing images: 10000


In [9]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channel, out_channel, stride = 1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channel, out_channel, kernel_size = 3, stride = stride, padding = 1, bias = False)
        self.bn1 = nn.BatchNorm2d(out_channel)
        self.conv2 = nn.Conv2d(out_channel, out_channel, kernel_size = 3, stride = 1, padding = 1, bias = False)
        self.bn2 = nn.BatchNorm2d(out_channel)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channel != out_channel:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channel, out_channel, kernel_size = 1, stride = stride, bias = False),
                nn.BatchNorm2d(out_channel)
            )
    
    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)   # Residual connection or Skip connection
        out = torch.relu(out)
        return out


Putting it into context with our MNIST example:
- Imagine a tensor moving from Stage 1 to Stage 2.
    - The Input ($x$): Enters the block as [Batch, 16, 28, 28].
    - The Main Path $F(x)$: Applies a stride of 2 and increases channels to 32. It outputs [Batch, 32, 14, 14].
    - The Problem: We cannot do [Batch, 32, 14, 14] + [Batch, 16, 28, 28]. It will throw a PyTorch dimension error.
    - The Solution: The code block triggers. The $1 \times 1$ convolution applies a stride of 2 (shrinking the $28 \times 28$ to $14 \times 14$) and expands the 16 channels to 32.
    - The Result: The shortcut successfully outputs a matched [Batch, 32, 14, 14] tensor, allowing the vital $F(x) + x$ addition to execute flawlessly.

In [10]:
class ResNet(nn.Module):
    def __init__(self):
        super(ResNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size = 3, stride = 1, padding = 1, bias = False)
        self.bn1 = nn.BatchNorm2d(16)

        self.layer1 = ResidualBlock(16,16, stride=1)
        self.layer2 = ResidualBlock(16,32, stride=2)
        self.layer3 = ResidualBlock(32,64, stride=2)

        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, 10)

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.avg_pool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out

In [ ]:
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ResNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr = 0.001)

num_epoch = 10

for epoch in range(num_epoch):
    model.train()
    total_loss = 0
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    avg_loss = total_loss / len(trainloader)

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in valloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    val_loss = criterion(outputs, labels).item()
    val_accuracy = correct / total
    print(f'Epoch [{epoch+1}/{num_epoch}], Loss: {avg_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}')


Epoch [1/10], Loss: 0.2992, Val Loss: 0.1923, Val Accuracy: 0.9744
Epoch [2/10], Loss: 0.0564, Val Loss: 0.0241, Val Accuracy: 0.9829
Epoch [3/10], Loss: 0.0418, Val Loss: 0.0222, Val Accuracy: 0.9853
Epoch [4/10], Loss: 0.0322, Val Loss: 0.0640, Val Accuracy: 0.9874
Epoch [5/10], Loss: 0.0268, Val Loss: 0.0589, Val Accuracy: 0.9882
Epoch [6/10], Loss: 0.0249, Val Loss: 0.0532, Val Accuracy: 0.9896
Epoch [7/10], Loss: 0.0208, Val Loss: 0.0269, Val Accuracy: 0.9900
Epoch [8/10], Loss: 0.0175, Val Loss: 0.0058, Val Accuracy: 0.9883
Epoch [9/10], Loss: 0.0162, Val Loss: 0.0447, Val Accuracy: 0.9843
Epoch [10/10], Loss: 0.0142, Val Loss: 0.0059, Val Accuracy: 0.9884


#### ResNet v2
In our code, the order of operations in the main path $F(x)$ was:

 Convolution -> BatchNorm -> ReLU

And at the very end of the block, we did:

 out = F(x) + x
 
 out = ReLU(out)

The authors realized that applying that final ReLU after the addition meant the skip connection path wasn't a perfect, unhindered superhighway. The gradient still had to pass through a ReLU gate during backpropagation.

The Fix: They moved the BatchNorm and ReLU to happen before the convolution weights.The new order is:
 
 BatchNorm -> ReLU -> Convolution.
 
Because the activation happens inside $F(x)$ before the math, the final addition becomes purely out = F(x) + x with no activation afterward. The skip connection becomes a mathematically pure identity path from the very first layer of the network straight to the last.